In [1]:
import pandas as pd
import numpy as np
import sqlalchemy 
from sqlalchemy import create_engine, text

#pd.set_option("display.max_columns", None)
#pd.set_option("display.max_rows", 100)
#pd.set_option("display.width", 1000)
#pd.set_option("display.max_colwidth", None)

In [2]:
df = pd.read_csv('../data/fraudTrain.csv')

In [3]:
df.rename(columns={'trans_date_trans_time': 'trans_datetime','age': 'customer_age',
                    'long': 'longitude', 'lat': 'latitude', 'first': 'first_name', 'last': 'last_name'}, inplace=True)

In [4]:
#Function to clean and optimize data types for memory efficiency while preserving analytical correctness.
def optimize_dtypes(df):
    """
    Transformations
    ---------------
    1. Remove unnecessary columns
    2. Convert low-cardinality text columns to category
    3. Convert columns to appropriate types
    4. Downcast numeric columns
    5. Report memory savings
    """

    # Memory Before
    memory_before = df.memory_usage(deep=True).sum() / (1024 ** 2)
    print("=" * 60)
    print(f"Memory Before Optimization : {memory_before:.2f} MB")
    print("=" * 60)

    # ==========================================================
    # Remove unnecessary columns
    if 'Unnamed: 0' in df.columns:
        df.drop(columns=['Unnamed: 0'], inplace=True)
        print("\n Dropped redundant index column 'Unnamed: 0'.") 

    # Data Transformation & Standardizing Types
    print("\n Transforming data types for database optimization")

    # To proper DateTime format
    df['trans_datetime'] = pd.to_datetime(df['trans_datetime'])
    df['dob'] = pd.to_datetime(df['dob'])

    # ZIP codes
    if "zip" in df.columns:
        df["zip"] = df["zip"].astype("string")

    # Remove the synthetic "fraud_" prefix from merchant names
    if "merchant" in df.columns:
        df["merchant"] = ( df["merchant"].str.replace("^fraud_", "", regex=True).astype("category"))

    # ==========================================================
    # Categorical Columns
    categorical_columns = [ "category", "first_name", "last_name", "gender",
                            "street", "city", "state", "job" ] 
   

    for col in categorical_columns:
        if col in df.columns:
            df[col] = df[col].astype("category")

    # ==========================================================
    # Float Columns
    float_columns = [ "amt", "latitude", "longitude", "merch_lat", "merch_long" ]

    for col in float_columns:
        if col in df.columns:
            df[col] = pd.to_numeric( df[col], downcast="float" )

    # ==========================================================
    # Integer Columns
    integer_columns = [ "city_pop", "is_fraud" ]

    for col in integer_columns:
        if col in df.columns:
            df[col] = pd.to_numeric( df[col], downcast="integer" )

    # ==========================================================
    # Memory After Optimization
    memory_after = df.memory_usage(deep=True).sum() / (1024 ** 2)

    reduction = ((memory_before - memory_after) / memory_before * 100 )

    print(f"Memory After Optimization  : {memory_after:.2f} MB")
    print(f"Memory Saved               : {memory_before - memory_after:.2f} MB")
    print(f"Reduction                  : {reduction:.2f}%")
    print("\n Data types successfully optimized.")

    print("=" * 60)

    return df

In [5]:
def data_summary(df):
    """
    Dataframe summary after optimization.
    """
    print("\nShape")
    print(df.shape)

    print("\nData Types")
    print(df.dtypes)

    print("\nMemory Usage")
    print(f"{df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

    print("\nMissing Values")
    print(df.isnull().sum().sum())

    print("\nDuplicate Rows")
    print(df.duplicated().sum())

In [6]:
df = optimize_dtypes(df)

Memory Before Optimization : 1034.99 MB

 Dropped redundant index column 'Unnamed: 0'.

 Transforming data types for database optimization
Memory After Optimization  : 261.21 MB
Memory Saved               : 773.78 MB
Reduction                  : 74.76%

 Data types successfully optimized.


In [7]:
data_summary(df)


Shape
(1296675, 22)

Data Types
trans_datetime    datetime64[ns]
cc_num                     int64
merchant                category
category                category
amt                      float64
first_name              category
last_name               category
gender                  category
street                  category
city                    category
state                   category
zip               string[python]
latitude                 float32
longitude                float32
city_pop                   int32
job                     category
dob               datetime64[ns]
trans_num                 object
unix_time                  int64
merch_lat                float32
merch_long               float32
is_fraud                    int8
dtype: object

Memory Usage
261.21 MB

Missing Values
0

Duplicate Rows
0


In [8]:
df['trans_hour'] = df['trans_datetime'].dt.hour.astype('int8') 

In [9]:
df['customer_age'] = ((df['trans_datetime'] - df['dob']).dt.days // 365).astype('int8')

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 24 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   trans_datetime  1296675 non-null  datetime64[ns]
 1   cc_num          1296675 non-null  int64         
 2   merchant        1296675 non-null  category      
 3   category        1296675 non-null  category      
 4   amt             1296675 non-null  float64       
 5   first_name      1296675 non-null  category      
 6   last_name       1296675 non-null  category      
 7   gender          1296675 non-null  category      
 8   street          1296675 non-null  category      
 9   city            1296675 non-null  category      
 10  state           1296675 non-null  category      
 11  zip             1296675 non-null  string        
 12  latitude        1296675 non-null  float32       
 13  longitude       1296675 non-null  float32       
 14  city_pop        12

In [43]:
df.to_csv('../data/transactions_clean.csv', index=False)

In [12]:
print(df.columns.tolist())

['trans_datetime', 'cc_num', 'merchant', 'category', 'amt', 'first_name', 'last_name', 'gender', 'street', 'city', 'state', 'zip', 'latitude', 'longitude', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud', 'trans_hour', 'customer_age']


In [11]:
df.head(10)

,trans_datetime,cc_num,merchant,category,amt,first_name,last_name,gender,street,city,...,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,trans_hour,customer_age
0,2019-01-01 00:00:18,2703186189652095,"Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,...,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011292,-82.048317,0,0,30
1,2019-01-01 00:00:44,630423337322,"Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,...,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159046,-118.186462,0,0,40
2,2019-01-01 00:00:51,38859492057661,Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,...,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150703,-112.154480,0,0,56
3,2019-01-01 00:01:16,3534093764340240,"Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,...,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034332,-112.561073,0,0,52
4,2019-01-01 00:03:06,375534208663984,Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,...,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632462,0,0,32
5,2019-01-01 00:04:08,4767265376804500,"Stroman, Hudson and Erdman",gas_transport,94.63,Jennifer,Conner,F,4655 David Island,Dublin,...,2158,Transport planner,1961-06-19,189a841a0a8ba03058526bcfe566aab5,1325376248,40.653381,-76.152664,0,0,57
6,2019-01-01 00:04:42,30074693890476,Rowe-Vandervort,grocery_net,44.54,Kelsey,Richards,F,889 Sarah Station Suite 624,Holcomb,...,2691,Arboriculturist,1993-08-16,83ec1cc84142af6e2acf10c44949e720,1325376282,37.162704,-100.153374,0,0,25
7,2019-01-01 00:05:08,6011360759745864,Corwin-Collins,gas_transport,71.65,Steven,Williams,M,231 Flores Pass Suite 720,Edinburg,...,6018,"Designer, multimedia",1947-08-21,6d294ed2cc447d2c71c7171a3d54967c,1325376308,38.948090,-78.540298,0,0,71
8,2019-01-01 00:05:18,4922710831011201,Herzog Ltd,misc_pos,4.27,Heather,Chase,F,6888 Hicks Stream Suite 954,Manor,...,1472,Public affairs consultant,1941-03-07,fc28024ce480f8ef21a32d64c93a29f5,1325376318,40.351814,-79.958145,0,0,77
9,2019-01-01 00:06:01,2720830304681674,"Schoen, Kuphal and Nitzsche",grocery_pos,198.39,Melissa,Aguilar,F,21326 Taylor Squares Suite 708,Clarksville,...,151785,Pathologist,1974-03-28,3b9014ea8fb80bd65de0b1463b00b00e,1325376361,37.179199,-87.485382,0,0,44
